# Azzaro: Whisper turbo vs small vs large

**Objetivo:** retranscribir el mismo video con los tres modelos y revisar los clips
donde producen textos distintos.

Esta version no usa las transcripciones anteriores. Los clips se forman con timestamps
por palabra y pausas reales, usando el mismo criterio de 3–10 segundos del pipeline.


## 1. Configuracion


In [6]:
from itertools import combinations
from pathlib import Path
import shutil
import sys

try:
    ROOT = Path.cwd().resolve()
except FileNotFoundError:
    ROOT = Path.home() / "labios-argentos"

while not (ROOT / "requirements.txt").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

if not (ROOT / "requirements.txt").exists():
    raise FileNotFoundError(
        "No se encontro la raiz del repo. Abrir labios-argentos y reiniciar el kernel."
    )

sys.path.insert(0, str(ROOT))

import pandas as pd
from IPython.display import Markdown, Video, display

from cleaning.visual_quality.src.whisper_model_comparison import (
    alinear_diferencias,
    construir_casos_por_clips_pipeline,
    descargar_video_yt,
    exportar_clips_revision_webm,
    extraer_palabras,
    resumen_diferencias,
    transcribir_whisper,
)

VIDEO_URL = "https://www.youtube.com/watch?v=a4ggqJZXnQE"
VIDEO_PATH = None

# En la Mac usamos MLX para aprovechar Apple Silicon.
BACKEND = "mlx"
MODELOS = ["turbo", "small", "large"]
MODELOS_MLX = {
    "turbo": "mlx-community/whisper-large-v3-turbo",
    "small": "mlx-community/whisper-small-mlx",
    "large": "mlx-community/whisper-large-v3-mlx",
}

# La corrida desde cero ya fue realizada en la Mac. False reutiliza exclusivamente
# los caches nuevos con timestamps por palabra; si no existen, se generan.
REPROCESAR_DESDE_CERO = False
MODELO_CORTE = "turbo"
COOKIES_FROM_BROWSER = "chrome"

# Seleccion manual realizada luego de revisar todos los clips corregidos.
# Son posiciones 1-based dentro de la lista completa de casos.
CASOS_SELECCIONADOS = [4, 5, 6, 10, 14, 24, 33, 34, 35, 42, 43]

WORKDIR = ROOT / "cleaning/visual_quality" / "outputs" / "azzaro_whisper"
ARTIFACTS_DIR = WORKDIR / "final"
TRANSCRIPTS_DIR = ARTIFACTS_DIR / "transcripts"
CLIPS_DIR = ARTIFACTS_DIR / "clips"
SELECTION_PATH = ARTIFACTS_DIR / "seleccion.csv"

# Compatibilidad con las corridas anteriores, que vivian bajo vsr/evaluation/.
LEGACY_WORKDIR = ROOT / "evaluation" / "outputs" / "azzaro_whisper"
LEGACY_TRANSCRIPTS_DIR = LEGACY_WORKDIR / "transcripts_word_timestamps_v2"
WORKDIR


PosixPath('/Users/martinbianchi/labios-argentos/cleaning/visual_quality/outputs/azzaro_whisper')

## 2. Cargar las transcripciones corregidas


In [7]:
if REPROCESAR_DESDE_CERO:
    if ARTIFACTS_DIR.exists():
        shutil.rmtree(ARTIFACTS_DIR)
elif not TRANSCRIPTS_DIR.exists() and LEGACY_TRANSCRIPTS_DIR.exists():
    TRANSCRIPTS_DIR.mkdir(parents=True, exist_ok=True)
    for cache in LEGACY_TRANSCRIPTS_DIR.glob("*__mlx__*.json"):
        shutil.copy2(cache, TRANSCRIPTS_DIR / cache.name)
    print("Caches nuevos migrados a la carpeta final versionable.")

caches_esperados = [
    TRANSCRIPTS_DIR / f"azzaro_racing_caracas__mlx__{modelo}.json"
    for modelo in MODELOS
]
artefactos_completos = (
    all(cache.exists() for cache in caches_esperados)
    and len(list(CLIPS_DIR.glob("*.webm"))) >= len(CASOS_SELECCIONADOS)
)

videos_locales = sorted((WORKDIR / "videos").glob("azzaro_racing_caracas.*"))
videos_locales += sorted((LEGACY_WORKDIR / "videos").glob("azzaro_racing_caracas.*"))
if VIDEO_PATH:
    video_path = Path(VIDEO_PATH).expanduser().resolve()
elif videos_locales:
    video_path = videos_locales[0]
elif artefactos_completos:
    # No se usa: los caches y clips finales ya estan versionados.
    video_path = WORKDIR / "videos" / "azzaro_racing_caracas.mp4"
else:
    video_path = descargar_video_yt(
        VIDEO_URL,
        WORKDIR / "videos",
        nombre_base="azzaro_racing_caracas",
        cookies_from_browser=COOKIES_FROM_BROWSER,
    )

resultados = {}
palabras = {}

for modelo in MODELOS:
    print(f"Cargando o transcribiendo {modelo} con {BACKEND}...")
    resultados[modelo] = transcribir_whisper(
        video_path,
        model_name=modelo,
        model_path=MODELOS_MLX[modelo] if BACKEND == "mlx" else None,
        output_dir=TRANSCRIPTS_DIR,
        backend=BACKEND,
        force=REPROCESAR_DESDE_CERO,
    )
    palabras[modelo] = extraer_palabras(resultados[modelo], modelo)

display(pd.DataFrame([
    {"modelo": modelo, "palabras_con_timestamp": len(palabras[modelo])}
    for modelo in MODELOS
]))


Cargando o transcribiendo turbo con mlx...
  cache valido: azzaro_racing_caracas__mlx__turbo.json
Cargando o transcribiendo small con mlx...
  cache valido: azzaro_racing_caracas__mlx__small.json
Cargando o transcribiendo large con mlx...
  cache valido: azzaro_racing_caracas__mlx__large.json


,modelo,palabras_con_timestamp
0,turbo,2565
1,small,2590
2,large,2820


## 3. Comparar sobre los clips corregidos


In [8]:
resumenes = []
for modelo_a, modelo_b in combinations(MODELOS, 2):
    diferencias = alinear_diferencias(
        palabras[modelo_a],
        palabras[modelo_b],
        contexto=0,
    )
    resumen = resumen_diferencias(
        diferencias,
        palabras[modelo_a],
        palabras[modelo_b],
    )
    resumenes.append({
        "comparacion": f"{modelo_a} vs {modelo_b}",
        "grupos_distintos": resumen["grupos_con_diferencias"],
        f"palabras_{modelo_a}": resumen["palabras_turbo_en_diferencias"],
        f"palabras_{modelo_b}": resumen["palabras_small_en_diferencias"],
    })

display(pd.DataFrame(resumenes))

# Turbo define los limites porque es el modelo usado por defecto en el pipeline.
# Cada texto se recupera despues usando exactamente el intervalo del clip.
casos = construir_casos_por_clips_pipeline(
    palabras,
    modelo_corte=MODELO_CORTE,
)

invalidos = [
    numero for numero in CASOS_SELECCIONADOS
    if numero < 1 or numero > len(casos)
]
if invalidos:
    raise IndexError(f"Casos seleccionados fuera de rango: {invalidos}")

casos_revision = [
    {**casos[numero - 1], "caso_seleccionado": numero}
    for numero in CASOS_SELECCIONADOS
]

print(f"Clips del pipeline donde los tres modelos difieren: {len(casos)}")
print(f"Clips seleccionados manualmente: {len(casos_revision)}")

columnas = [
    "caso_seleccionado",
    "clip_id",
    "start",
    "end",
    "transcripcion_turbo",
    "transcripcion_small",
    "transcripcion_large",
]
tabla_seleccion = pd.DataFrame(casos_revision)[columnas]
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
tabla_seleccion.to_csv(SELECTION_PATH, index=False)
display(tabla_seleccion)
print(f"Seleccion guardada en: {SELECTION_PATH}")


,comparacion,grupos_distintos,palabras_turbo,palabras_small,palabras_large
0,turbo vs small,128,182.0,207.0,NaN
1,turbo vs large,69,158.0,NaN,413.0
2,small vs large,147,NaN,232.0,462.0


Clips del pipeline donde los tres modelos difieren: 53
Clips seleccionados manualmente: 11


,caso_seleccionado,clip_id,start,end,transcripcion_turbo,transcripcion_small,transcripcion_large
0,4,11,93.64,98.50,Estoy convencido de que él lo debe sufrir hast...,convencido de que él lo debe sufrir hasta incl...,Estoy convencido de que él lo debe sufrir hast...
1,5,13,109.58,113.94,"Entonces, por más que me duela por lo que es c...",entonces por más que me duele por lo que es co...,"Entonces, por más que me digan que no, no lo s..."
2,6,17,136.96,147.36,Entraste porque... De milagro estaba adelantad...,porque de milagro estaba adelantado un hombro ...,Entraste porque de milagro estaba adelantado.....
3,10,29,244.26,254.58,A Milito lo vamos a criticar. Pero hasta el úl...,Milito lo vamos a criticar pero hasta el últim...,"Milito lo vamos a criticar, pero hasta el últi..."
4,14,50,411.30,421.58,Kostal no quiere poner a Vergara refuerzo. Con...,no quiere poner a bregar refuerzo con Ignis su...,no quiere poner a Vergara refuerzo. Con él ni ...
5,24,83,692.74,701.24,¿Qué los voy a empezar a nombrar ahora? Vergar...,los voy a empezar a nombrar ahora a ver qué no...,lo voy a empezar a nombrar ahora verdad no pue...
6,33,94,774.34,784.62,Que se banque. Que lo van a putear a Milito. P...,que se banque que lo van a putear a Medito por...,se banque que lo van a putear a milito pues yo...
7,34,95,785.44,793.22,Porque para gobernar. Hay que tener pelotas. L...,porque para gobernar hay que tener pelotas las...,para gobernar hay que tener peludo y no hay qu...
8,35,97,801.00,811.22,Que este es. Su Racing en parte. Pero entienda...,este es su Racing en parte pero entiendan perf...,sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí s...
9,42,117,948.14,953.94,No. No. No. No. No. No. No.,me gole que ramaravilla la puta que lo parió p...,"Me voy a querer la maravilla, la puta que lo p..."


Seleccion guardada en: /Users/martinbianchi/labios-argentos/cleaning/visual_quality/outputs/azzaro_whisper/final/seleccion.csv


## 4. Escuchar los casos relevantes

Los clips se regeneran en WebM para que VSCode pueda reproducir video y audio. Cada
transcripción corresponde al clip completo mostrado, no a un tramo vecino.


In [ ]:
clips = exportar_clips_revision_webm(
    video_path,
    casos_revision,
    CLIPS_DIR,
    margen=0.08,
    force=False,
)

for i, caso in enumerate(clips, start=1):
    display(Markdown(
        f"### Caso {i} | "
        f"clip_{int(caso['clip_id']):04d} | "
        f"{caso['clip_start']}s-{caso['clip_end']}s"
    ))

    display(Video(
        filename=caso["clip_path"],
        embed=True,
        mimetype="video/webm",
        html_attributes="controls preload='metadata'",
    ))

    print("turbo:", caso["transcripcion_turbo"])
    print("small:", caso["transcripcion_small"])
    print("large:", caso["transcripcion_large"])


### Caso 1 | clip_0011 | 93.56s-98.58s

turbo: Estoy convencido de que él lo debe sufrir hasta incluso más que cualquier raci. Racingista.
small: convencido de que él lo debe sufrir hasta incluso más que cualquier racingista
large: Estoy convencido de que él lo debe sufrir hasta incluso más que cualquier racinguista.


### Caso 2 | clip_0013 | 109.5s-114.02s

turbo: Entonces, por más que me duela por lo que es costas...
small: entonces por más que me duele por lo que es costas
large: Entonces, por más que me digan que no, no lo sé. Me duela por lo que es Costas.


### Caso 3 | clip_0017 | 136.88s-147.44s

...

turbo: Que este es. Su Racing en parte. Pero entiendan perfectamente. A donde estoy yendo. Ya está. No hay excusas. Ahora armá tu equipo. Con tu técnico.
small: este es su Racing en parte pero entiendan perfectamente a donde estoy yendo ya estaba no hay excusa ahora armad tu equipo con tu técnico
large: sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí sí


### Caso 10 | clip_0117 | 948.06s-954.02s

turbo: No. No. No. No. No. No. No.
small: me gole que ramaravilla la puta que lo parió pero gente
large: Me voy a querer la maravilla, la puta que lo parió. Pero gente,


### Caso 11 | clip_0119 | 960.58s-964.0s

turbo: No. No. No.
small: mal y lo vengo contando hace tiempo que racing está haciendo
large: mal. Y lo vengo contando hace tiempo que Racing está haciendo las
